In [2]:
from selenium import webdriver
from bs4 import BeautifulSoup
import json
import time


fraud_urls = (
('網路購物詐騙', 'https://165dashboard.tw/fraud-method/329818012855046144'),
('假投資詐騙', 'https://165dashboard.tw/fraud-method/327303345054158848'),
('釣魚連結詐騙', 'https://165dashboard.tw/fraud-method/320803405264064512'),
('假中獎通知詐騙', 'https://165dashboard.tw/fraud-method/322206935585656832'),
('假求職詐騙', 'https://165dashboard.tw/fraud-method/325471681525059584'),
('假借銀行貸款詐騙', 'https://165dashboard.tw/fraud-method/270466355268947968'),
('騙取金融帳戶(卡片)詐騙', 'https://165dashboard.tw/fraud-method/320399472645378048'),
('猜猜我是誰詐騙', 'https://165dashboard.tw/fraud-method/307801223623348224'),
('假慈善機關(急難救助)詐騙', 'https://165dashboard.tw/fraud-method/289656464379744256'),
('收購郵銀簿(電信門號)詐騙', 'https://165dashboard.tw/fraud-method/338873881450254336'),
('六合彩詐騙', 'https://165dashboard.tw/fraud-method/286781286448631808'),
('信用卡遭盜刷詐騙', 'https://165dashboard.tw/fraud-method/336010695290130432'),
('假交友(投資詐財)詐騙', 'https://165dashboard.tw/fraud-method/312064826387992576'),
('色情應召詐財詐騙', 'https://165dashboard.tw/fraud-method/321136076590682112'),
('假檢警詐騙', 'https://165dashboard.tw/fraud-method/325103882076164096'),
('假買家騙賣家詐騙', 'https://165dashboard.tw/fraud-method/334882749754118144'),
('假交友(徵婚詐財)詐騙', 'https://165dashboard.tw/fraud-method/332367662917095424'),
('假廣告詐騙', 'https://165dashboard.tw/fraud-method/328005776549351424'),
('盜(冒)用 LINE 帳號詐騙', 'https://165dashboard.tw/fraud-method/205961288457654272'),
('虛擬遊戲詐騙', 'https://165dashboard.tw/fraud-method/330543633121939456'),
('假預付型消費詐騙', 'https://165dashboard.tw/fraud-method/343281639280873472'),
('假推銷詐騙', 'https://165dashboard.tw/fraud-method/223628654511394816'),
('假快遞詐騙', 'https://165dashboard.tw/fraud-method/184119756654120960')
)


options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    options=options
)


all_results = []


for index, (tag_name, url) in enumerate(
    fraud_urls,
    start=1
):

    driver.get(url)

    time.sleep(4)

    html = driver.page_source

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # -----------------------------
    # 抓防詐小撇步 / 詐騙話術
    # -----------------------------

    detail_results = []

    detail_cards = soup.find_all(
        "div",
        class_="detail-card"
    )

    for card in detail_cards:

        title = card.find(
            "div",
            class_="detail-card__title"
        )

        content = card.find(
            "div",
            class_="detail-card__content"
        )

        if title and content:

            title_text = title.get_text(
                " ",
                strip=True
            )

            content_text = content.get_text(
                "\n",
                strip=True
            )

            detail_results.append(
            {
             "title": title_text,
             "content": content_text
            }
    )


    # -----------------------------
    # 抓案例
    # -----------------------------

    case_title = ""
    case_content = ""

    case_details = soup.find_all(
        "div",
        class_="case-detail__content"
    )

    for case_detail in case_details:

        title_block = case_detail.find(
            "div",
            class_="title-block"
        )

        content_block = case_detail.find(
            "div",
            class_="content-block"
        )

        if not content_block:
            continue

        if title_block:
            title = title_block.find(
                "div",
                class_="title"
            )
        else:
            title = None

        if not title:
            continue

        current_title = title.get_text(
            " ",
            strip=True
        )

        current_content = content_block.get_text(
            "\n",
            strip=True
        )

        if not current_content:
            continue

        case_title = current_title
        case_content = current_content

        break


    # -----------------------------
    # 建立資料
    # -----------------------------

    result = {

        "tag_name": tag_name,

        "url": url,

        "details": detail_results,

        "case_title": case_title,

        "case_content": case_content

    }

    all_results.append(result)


driver.quit()


# ============================================================
# 儲存 JSON
# ============================================================

with open(
    "fraud_data.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        ensure_ascii=False,
        indent=4
    )


print("=" * 60)
print("JSON 儲存完成")
print("=" * 60)
print(f"資料筆數：{len(all_results)}")

JSON 儲存完成
資料筆數：23


In [3]:
import json

with open("fraud_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("資料筆數：", len(data))
print("\n第一筆資料：")
print(json.dumps(data[0], ensure_ascii=False, indent=4))

資料筆數： 23

第一筆資料：
{
    "tag_name": "網路購物詐騙",
    "url": "https://165dashboard.tw/fraud-method/329818012855046144",
    "details": [
        {
            "title": "防詐小撇步",
            "content": "1、\t賣貨便、物流平台不會要求加入私人 LINE 客服處理交易。\n2、\t注意網址與官方網站是否一致，若發現異狀，應立即停止操作。\n3、\t網路購物不需要透過網銀轉帳來完成「認證」或「帳戶重啟」。\n4、\t無卡提款碼、網銀操作都是金流授權行為，切勿依陌生人指示操作。\n5、\t遇到購物糾紛或系統問題，應直接透過官方客服查證。"
        },
        {
            "title": "詐騙話術解析",
            "content": "1、\t詐騙集團在社群平台刊登熱門商品吸引買家上鉤。\n2、\t架設仿冒網站，讓被害人誤以為進入官方交易系統。\n3、\t佯稱系統異常、實名認證未完成等理由，誘導加入假客服 LINE，由專人一步步操控被害人。\n4、\t謊稱進行帳戶驗證，實際上是在引導轉帳及提款。"
        }
    ],
    "case_title": "誤入詐騙戰鬥盤　積蓄慘遭爆裂衝擊",
    "case_content": "某天在家裡滑手機，知名社群平台上看到一個帳號貼文販售新款戰鬥陀螺。我曾眼睜睜看著這款陀螺在電商平台被秒殺，自己也無暇到實體店面徹夜排隊購買，如今眼前又出現入手機會，不想再扼腕，馬上留言表明購買意願。\n沒多久，對方私訊回復傳來一組網址，聲稱是7-11賣貨便交易連結。由於我過去也曾有賣貨便購物經驗，不疑有他，直接點進連結填寫收件與聯絡資料。\n資料填完後，頁面突然跳出訊息，顯示我尚未完成「實名認證」，無法完成交易。接著畫面又出現一個 LINE 客服連結，要我加入客服處理認證問題。\n我加入客服人員後，對方立即回訊表示，只要配合進行帳戶重啟程序，就不需要重新驗證身分，可以直接完成交易。客服先要求我打開網路銀行APP，然後指示我各種花式操作，強調驗證程序不會真的扣款。\n當時我雖有些疑惑，一心只

In [4]:
print(json.dumps(data[0], ensure_ascii=False, indent=4))

{
    "tag_name": "網路購物詐騙",
    "url": "https://165dashboard.tw/fraud-method/329818012855046144",
    "details": [
        {
            "title": "防詐小撇步",
            "content": "1、\t賣貨便、物流平台不會要求加入私人 LINE 客服處理交易。\n2、\t注意網址與官方網站是否一致，若發現異狀，應立即停止操作。\n3、\t網路購物不需要透過網銀轉帳來完成「認證」或「帳戶重啟」。\n4、\t無卡提款碼、網銀操作都是金流授權行為，切勿依陌生人指示操作。\n5、\t遇到購物糾紛或系統問題，應直接透過官方客服查證。"
        },
        {
            "title": "詐騙話術解析",
            "content": "1、\t詐騙集團在社群平台刊登熱門商品吸引買家上鉤。\n2、\t架設仿冒網站，讓被害人誤以為進入官方交易系統。\n3、\t佯稱系統異常、實名認證未完成等理由，誘導加入假客服 LINE，由專人一步步操控被害人。\n4、\t謊稱進行帳戶驗證，實際上是在引導轉帳及提款。"
        }
    ],
    "case_title": "誤入詐騙戰鬥盤　積蓄慘遭爆裂衝擊",
    "case_content": "某天在家裡滑手機，知名社群平台上看到一個帳號貼文販售新款戰鬥陀螺。我曾眼睜睜看著這款陀螺在電商平台被秒殺，自己也無暇到實體店面徹夜排隊購買，如今眼前又出現入手機會，不想再扼腕，馬上留言表明購買意願。\n沒多久，對方私訊回復傳來一組網址，聲稱是7-11賣貨便交易連結。由於我過去也曾有賣貨便購物經驗，不疑有他，直接點進連結填寫收件與聯絡資料。\n資料填完後，頁面突然跳出訊息，顯示我尚未完成「實名認證」，無法完成交易。接著畫面又出現一個 LINE 客服連結，要我加入客服處理認證問題。\n我加入客服人員後，對方立即回訊表示，只要配合進行帳戶重啟程序，就不需要重新驗證身分，可以直接完成交易。客服先要求我打開網路銀行APP，然後指示我各種花式操作，強調驗證程序不會真的扣款。\n當時我雖有些疑惑，一心只想趕緊入手新款陀螺，就這樣轉出數萬

In [5]:
import sqlite3

# 建立 / 連線 SQLite 資料庫
conn = sqlite3.connect("fraud.db")

# 建立 cursor，用來執行 SQL
cursor = conn.cursor()

print("資料庫連線成功")

資料庫連線成功


In [6]:
# ============================================================
# 建立 fraud_types
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS fraud_types (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    url TEXT NOT NULL UNIQUE
)
""")


# ============================================================
# 建立 fraud_details
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS fraud_details (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    fraud_type_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    content TEXT NOT NULL,

    FOREIGN KEY (fraud_type_id)
        REFERENCES fraud_types(id)
        ON DELETE CASCADE
)
""")


# ============================================================
# 建立 fraud_cases
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS fraud_cases (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    fraud_type_id INTEGER NOT NULL,
    title TEXT,
    content TEXT,

    FOREIGN KEY (fraud_type_id)
        REFERENCES fraud_types(id)
        ON DELETE CASCADE
)
""")


conn.commit()

print("資料表建立完成")

資料表建立完成


In [7]:
import json

with open(
    "fraud_data.json",
    "r",
    encoding="utf-8"
) as f:

    fraud_data = json.load(f)


print("JSON 資料筆數：", len(fraud_data))

JSON 資料筆數： 23


In [8]:
for fraud in fraud_data:

    # ========================================================
    # 1. fraud_types
    # ========================================================

    cursor.execute(
        """
        INSERT OR IGNORE INTO fraud_types (
            name,
            url
        )
        VALUES (?, ?)
        """,
        (
            fraud["tag_name"],
            fraud["url"]
        )
    )


    # ========================================================
    # 找到該詐騙類型的 id
    # ========================================================

    cursor.execute(
        """
        SELECT id
        FROM fraud_types
        WHERE url = ?
        """,
        (
            fraud["url"],
        )
    )

    fraud_type_id = cursor.fetchone()[0]


    # ========================================================
    # 2. fraud_details
    # ========================================================

    for detail in fraud["details"]:

        cursor.execute(
            """
            INSERT INTO fraud_details (
                fraud_type_id,
                title,
                content
            )
            VALUES (?, ?, ?)
            """,
            (
                fraud_type_id,
                detail["title"],
                detail["content"]
            )
        )


    # ========================================================
    # 3. fraud_cases
    # ========================================================

    if fraud["case_title"] or fraud["case_content"]:

        cursor.execute(
            """
            INSERT INTO fraud_cases (
                fraud_type_id,
                title,
                content
            )
            VALUES (?, ?, ?)
            """,
            (
                fraud_type_id,
                fraud["case_title"],
                fraud["case_content"]
            )
        )


# 儲存變更
conn.commit()

print("JSON 資料已成功寫入 SQLite")

JSON 資料已成功寫入 SQLite


In [10]:
cursor.execute("""
SELECT *
FROM fraud_types
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

print("詐騙類型數量：", len(rows))

(1, '網路購物詐騙', 'https://165dashboard.tw/fraud-method/329818012855046144')
(2, '假投資詐騙', 'https://165dashboard.tw/fraud-method/327303345054158848')
(3, '釣魚連結詐騙', 'https://165dashboard.tw/fraud-method/320803405264064512')
(4, '假中獎通知詐騙', 'https://165dashboard.tw/fraud-method/322206935585656832')
(5, '假求職詐騙', 'https://165dashboard.tw/fraud-method/325471681525059584')
(6, '假借銀行貸款詐騙', 'https://165dashboard.tw/fraud-method/270466355268947968')
(7, '騙取金融帳戶(卡片)詐騙', 'https://165dashboard.tw/fraud-method/320399472645378048')
(8, '猜猜我是誰詐騙', 'https://165dashboard.tw/fraud-method/307801223623348224')
(9, '假慈善機關(急難救助)詐騙', 'https://165dashboard.tw/fraud-method/289656464379744256')
(10, '收購郵銀簿(電信門號)詐騙', 'https://165dashboard.tw/fraud-method/338873881450254336')
(11, '六合彩詐騙', 'https://165dashboard.tw/fraud-method/286781286448631808')
(12, '信用卡遭盜刷詐騙', 'https://165dashboard.tw/fraud-method/336010695290130432')
(13, '假交友(投資詐財)詐騙', 'https://165dashboard.tw/fraud-method/312064826387992576')
(14, '色情應召詐財詐騙', 'h

In [11]:
tables = [
    "fraud_types",
    "fraud_details",
    "fraud_cases"
]

for table in tables:

    cursor.execute(
        f"SELECT COUNT(*) FROM {table}"
    )

    count = cursor.fetchone()[0]

    print(
        f"{table}: {count} 筆"
    )

fraud_types: 23 筆
fraud_details: 46 筆
fraud_cases: 23 筆


In [12]:
cursor.execute("""
SELECT
    fraud_types.name,
    fraud_details.title,
    fraud_details.content

FROM fraud_types

JOIN fraud_details
ON fraud_types.id = fraud_details.fraud_type_id

WHERE fraud_types.name = ?
""",
(
    "網路購物詐騙",
)
)

results = cursor.fetchall()

for row in results:

    print("=" * 60)

    print("詐騙類型：")
    print(row[0])

    print("\n資料類型：")
    print(row[1])

    print("\n內容：")
    print(row[2])

詐騙類型：
網路購物詐騙

資料類型：
防詐小撇步

內容：
1、	賣貨便、物流平台不會要求加入私人 LINE 客服處理交易。
2、	注意網址與官方網站是否一致，若發現異狀，應立即停止操作。
3、	網路購物不需要透過網銀轉帳來完成「認證」或「帳戶重啟」。
4、	無卡提款碼、網銀操作都是金流授權行為，切勿依陌生人指示操作。
5、	遇到購物糾紛或系統問題，應直接透過官方客服查證。
詐騙類型：
網路購物詐騙

資料類型：
詐騙話術解析

內容：
1、	詐騙集團在社群平台刊登熱門商品吸引買家上鉤。
2、	架設仿冒網站，讓被害人誤以為進入官方交易系統。
3、	佯稱系統異常、實名認證未完成等理由，誘導加入假客服 LINE，由專人一步步操控被害人。
4、	謊稱進行帳戶驗證，實際上是在引導轉帳及提款。
